## 필독!!!

<h3> 여기 있는 코드는 절대 실행하지 마십시오. </h3>

눈으로만 보고 이해하시거나

복붙하셔서 실제 linux나 파이썬 환경에서 실행해 주시기 바랍니다.

이 파일은 jupyter 파일입니다.

여기 있는 코드는 모두 jupyter가 아닌 실제 파이썬 및 ROS2 환경에서 사용할 수 있는 코드로 작성하였습니다.

ROS2 코드를 jupyter에서 실행하는 방법이 없는 것은 아니나 별도의 방법이 따로 존재하기 때문에(`jupyter_ws` 참고)

여기 있는 코드를 실행하게 될 경우 일부 오류나 무한루프 등에 빠질 수 있는 위험이 있습니다.

### Launch 패키지 실습

#### 1 - 1. 노드파일 확인 (talker_node.py) 

(py_launch_example/py_launch_example/talker_node.py 참고.)

In [ ]:
import rclpy
from rclpy.node import Node
from std_msgs.msg import String

class TalkerNode(Node):
    def __init__(self):
        super().__init__('talker_node')
        self.publisher = self.create_publisher(String, 'chatter', 10)
        self.timer = self.create_timer(1.0, self.timer_callback)
        self.count = 0

    def timer_callback(self):
        msg = String()
        msg.data = f'Hello ROS2 Launch {self.count}'
        self.publisher.publish(msg)
        self.get_logger().info(f'Publish: {msg.data}')
        self.count += 1

def main(args=None):
    rclpy.init(args=args)
    node = TalkerNode()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        node.destroy_node()
        rclpy.shutdown()

if __name__ == '__main__':
    main()

#### 1 - 2. 노드파일 확인 (listener_node.py) 

(py_launch_example/py_launch_example/listener_node.py 참고.)

In [ ]:
import rclpy
from rclpy.node import Node
from std_msgs.msg import String

class ListenerNode(Node):
    def __init__(self):
        super().__init__('listener_node')
        self.subscription = self.create_subscription(
            String,
            'chatter',
            self.listener_callback,
            10
        )
    def listener_callback(self, msg):
        self.get_logger().info(f'Receive: {msg.data}')

def main(args=None):
    rclpy.init(args=args)
    node = ListenerNode()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        node.destroy_node()
        rclpy.shutdown()

if __name__ == '__main__':
    main()

#### 1 - 3. 빌드 및 실행

setup.py 내용 추가하고 빌드 작업.

In [ ]:
entry_points={
    'console_scripts': [
        'talker_node = py_launch_example.talker_node:main',
        'listener_node = py_launch_example.listener_node:main',
    ],
}

터미널 2개 띄운 후 양쪽 모두 

```bash
source install/setup.bash
```

이후 각각 아래 명령어를 한개씩 실행한다.

```bash
ros2 run py_launch_example talker_node
ros2 run py_launch_example listener_node
```

송수신이 잘 되는지 확인한다.

#### 2 - 1. launch파일 확인 (talker_listener.launch.py) 

(py_launch_example/launch/talker_listener.launch.py 참고.)

In [ ]:
from launch import LaunchDescription
from launch_ros.actions import Node

def generate_launch_description():
    talker_node = Node(
        package='py_launch_example',
        executable='talker_node',
        name='talker'
    )

    listener_node = Node(
        package='py_launch_example',
        executable='listener_node',
        name='listener'
    )

    return LaunchDescription([
        talker_node,
        listener_node
    ])

In [ ]:
from launch import LaunchDescription
from launch_ros.actions import Node

ROS2 launch 파일에서 실행할 항목들을 묶어서 반환하는 클래스 `LaunchDescription`과 ROS2 노드를 실행하기 위한 `Node` 클래스를 불러온다.

In [ ]:
def generate_launch_description():
    talker_node = Node(
        package='py_launch_example',
        executable='talker_node',
        name='talker'
    )

    listener_node = Node(
        package='py_launch_example',
        executable='listener_node',
        name='listener'
    )

`py_launch_example` 패키지의 `talker_node`(별칭. 노드이름 아님.) 노드를 실행하게 설정.

그리고 실행할 노드 이름을 `talker`로 설정한다.

사실상 아래 실행명령어와 같다.

```bash
ros2 run py_launch_example talker_node
```

listener_node는 구조가 완전히 동일하므로 생략.

\#\# 주의사항

여기서 함수로 쓰인 `generate_launch_description()`은 반드시 이 이름으로만 사용해야 하며, 다른 이름으로 함수를 정의하면 안된다.

이유는 

```bash
ros2 launch py_launch_example talker_listener.launch.py
```

로 launch 명령어를 실행하면 

ros2 launch 명령어가 먼저 실행되고, 이 명령어가 해당 패키지 안에서 launch 파일을 찾은 다음,

내부적으로는 아래 과정처럼 동작하는데

```python
# 1. launch 파일 경로 찾기
launch_file_path = ".../py_launch_example/launch/talker_listener.launch.py"

# 2. 그 파일을 파이썬 모듈처럼 읽어옴
module = import_launch_file(launch_file_path)

# 3. 그 모듈 안에서 정해진 이름의 함수 찾기
func = module.generate_launch_description

# 4. 함수 호출
launch_description = func()

# 5. 반환된 LaunchDescription 안의 액션 실행
launch_service.run(launch_description)
```

여기서 호출하는 함수 이름이 `generate_launch_description()`이기 때문에

다른 이름으로 함수를 정의하면 찾지 못해 실행될 수 없다.

In [ ]:
return LaunchDescription([
    talker_node,
    listener_node
])

launch 시스템에게 실행할 항목 목록을 반환하는 코드.

즉 위에서 정의한 talker_node, listener_node를 실행하라는 코드이다.

#### 빌드 및 실행 (launch 파일로 노드 실행하기)

setup.py에 아래처럼 한 줄(가독성을 위해 2줄로 나눠 작성했다.)이 추가되었는지 확인한다.

In [ ]:
data_files=[
    ('share/ament_index/resource_index/packages',
        ['resource/' + package_name]),
    ('share/' + package_name,
        ['package.xml']),
    (
        os.path.join('share', package_name, 'launch'),                  # 추가된 한줄
        glob('launch/*.launch.py')                                      # 추가된 한줄
    ),
],

또한 추가된 줄은 os모듈과 glob 모듈을 사용하기 때문에

setup.py 맨 위에 아래 두줄도 추가.

```python
import os
from glob import glob
````

수정 후 빌드를 해준다.

이번에는 터미널을 1개만 띄워도 된다.

```bash
source install/setup.bash
```

이후 아래 명령어를 실행하고 결과를 확인한다.

```bash
ros2 launch py_launch_example talker_listener.launch.py
```

송수신 내용이 터미널 하나에 모두 잘 출력되는지 확인한다.

#### 2 - 2. launch파일 확인 (include_launch.launch.py) 

(py_launch_example/launch/include_launch.launch.py 참고.)

In [ ]:
from launch import LaunchDescription
from launch.actions import IncludeLaunchDescription
from launch.launch_description_sources import PythonLaunchDescriptionSource
from ament_index_python.packages import get_package_share_directory
import os

def generate_launch_description():
    included_launch = IncludeLaunchDescription(
        PythonLaunchDescriptionSource(
            os.path.join(
                get_package_share_directory('py_launch_example'),
                'launch',
                'talker_listener.launch.py'
            )
        )
    )

    return LaunchDescription([
        included_launch
    ])

In [ ]:
from launch import LaunchDescription
from launch.actions import IncludeLaunchDescription
from launch.launch_description_sources import PythonLaunchDescriptionSource
from ament_index_python.packages import get_package_share_directory
import os

`from launch.actions import IncludeLaunchDescription` : 다른 launch 파일을 현재 launch에 포함하여 실행하는 Action 클래스

`from launch.launch_description_sources import PythonLaunchDescriptionSource` : 포함할 launch 파일이 '*.launch.py'형태라고 알려주는 클래스.

ROS2에는 launch 파일이 python, xml, yaml 등 다양한 파일 형식들이 있기 때문에

'이 launch 파일은 파이썬 파일 형식입니다'라고 알려주는 데 사용하는 코드이다.

`from ament_index_python.packages import get_package_share_directory` : 패키지의 share 폴더 경로를 찾아주는 함수.

`get_package_share_directory('py_launch_example')`는 install 폴더에서 'py_launch_example' 패키지의 경로를 반환하고,

이는 일반적으로 install/py_launch_example/share/py_launch_example 이 경로가 나온다.


In [ ]:
def generate_launch_description():
    included_launch = IncludeLaunchDescription(
        PythonLaunchDescriptionSource(
            os.path.join(
                get_package_share_directory('py_launch_example'),
                'launch',
                'talker_listener.launch.py'
            )
        )
    )

`IncludeLaunchDescription` : 다른 launch 파일을 현재 launch에 포함하여 실행하는 Action 클래스

`PythonLaunchDescriptionSource` : 포함할 launch 파일이 Python 형태의 launch 파일이라고 알려주는 클래스.

`os.path.join(get_package_share_directory('py_launch_example'), 'launch', 'talker_listener.launch.py')` : 런치파일 경로 추가

install/py_launch_example/share/py_launch_example/launch/talker_listener.launch.py 경로.

\#\# 주의사항

여기서도 함수로 쓰인 `generate_launch_description()`은 반드시 이 이름으로만 사용해야 하며, 다른 이름으로 함수를 정의하면 안된다.

#### 빌드 및 실행 (launch 파일로 노드 실행하기)

setup.py에 `(os.path.join('share', package_name, 'launch'), glob('launch/*.launch.py'))` 줄이 추가되어있는지 확인하고 없으면 넣어준다.

그리고 빌드한다.

```bash
source install/setup.bash
```

를 실행한 후 아래 명령어를 실행하고 결과를 확인한다.

```bash
ros2 launch py_launch_example include_launch.launch.py
```

송수신 내용이 터미널 하나에 모두 잘 출력되는지 확인한다.

#### 2 - 3. launch파일 확인 (namespace_remap.launch.py) 

(py_launch_example/launch/namespace_remap.launch.py 참고.)

In [ ]:
from launch import LaunchDescription
from launch_ros.actions import Node

def generate_launch_description():
    talker_node = Node(
        package='py_launch_example',
        executable='talker_node',
        name='talker',
        namespace='robot1',
        remappings=[
            ('chatter', 'robot_chatter')
        ]
    )

    listener_node = Node(
        package='py_launch_example',
        executable='listener_node',
        name='listener',
        namespace='robot1',
        remappings=[
            ('chatter', 'robot_chatter')
        ]
    )

    return LaunchDescription([
        talker_node,
        listener_node
    ])

In [ ]:
def generate_launch_description():
    talker_node = Node(
        package='py_launch_example',
        executable='talker_node',
        name='talker',
        namespace='robot1',
        remappings=[
            ('chatter', 'robot_chatter')
        ]
    )

`package='py_launch_example'` : 패키지 이름

`executable='talker_node'` : 실행할 노드

`name='talker'` : 노드 이름

`namespace='robot1'` : 네임스페이스. 즉 이것이 있으면 노드 이름은 `talker`가 아닌 `robot1/talker`가 된다.

`remappings=[('chatter', 'robot_chatter')]` : 코드 안에서 쓰는 토픽 이름 chatter를 launch 실행 시 robot_chatter로 바꿈.

즉, 이 remap launch를 사용하는 이유는 `코드의 추가수정 없이` 토픽을 변경해서 쓰기 위함이다.

여기서도 마찬가지로 `generate_launch_description()` 이름을 바꾸면 안된다.

나머지 부분은 동일하므로 생략.

#### 빌드 및 실행 (launch 파일로 노드 실행하기)

setup.py에 `(os.path.join('share', package_name, 'launch'), glob('launch/*.launch.py'))` 줄이 추가되어있는지 확인하고 없으면 넣어준다.

그리고 빌드한다.

이번엔 터미널 2개를 준비하고 양쪽 모두

```bash
source install/setup.bash
```

를 실행한 후 한쪽 터미널에 아래 명령어를 실행하고 결과를 확인한다.

```bash
ros2 launch py_launch_example namespace_remap.launch.py
```

송수신 내용이 터미널 하나에 모두 잘 출력되는지 확인한다.

이후 반대쪽 터미널에 

```bash
ros2 node list
```

를 실행하여 /robot1/listener, /robot1/talker 노드가 뜨는지 확인하고

```bash
ros2 topic list
```

를 실행하여 /robot1/robot_chatter 토픽이 뜨는지 확인한다.

### ROS2 CLI 명령어 학습하기

[ROS2_CLI](../ros2기초/ROS2_CLI.md)